# Notebook for causal analysis for the AGE subgroups using DoWhy-library

In [ ]:
from dowhy import CausalModel
import pandas as pd

### Define outcome and the confounders for each feature

In [ ]:
outcome = "How happy are you?"

In [ ]:
feature_confounder_map = {
    "Health condition": [
        "Age",
        "Income quartiles",
        "Chronic health problems?",
        "Education completed",
        "Employment - 7 groups"
    ],

    "I generally feel that what I do in life is worthwhile": [
        "A person to get support from when feeling depressed",
        "How frequently participate in social activities?",
        "Employment - 7 groups",
        "Marital status",
        "Health condition"
    ],

    "Can't find the way because life has become so complicated?": [
        "Education completed",
        "Employment - 7 groups",
        "A person to get support from when feeling depressed",
        "Age",
        "Household size"
    ],

    "I feel I am free to decide how to live my life": [
        "Income quartiles",
        "Education completed",
        "How much trust the government?"
    ],

    "I am optimistic about the future": [
        "Personal financial situation",
        "Access to recreational or green areas?",
        "A person to get support from to raise emergency money",
        "Health condition",
        "Age",
        "Employment - 7 groups"
    ],

    "Household able to make ends meet?": [
        "Employment - 7 groups",
        "Income quartiles",
        "Personal financial situation",
        "Household size",
        "No. of children"
    ],

    "Personal financial situation": [
        "Employment - 7 groups",
        "Can afford a meal with meat/chicken/fish every second day?",
        "Household structure",
        "Income quartiles",
        "Education completed"
    ],

    "How much trust the police?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the government?",
        "How much trust the legal system?",
        "Rural/urban living",
        "Age"
    ],

    "Deprivation index: No. of items hhold can't afford": [
        "Income quartiles",
        "Employment - 7 groups",
        "Can afford to keep home adequately warm?",
        "Household size",
        "Household structure"
    ],

    "Quality of education system?": [
        "Education completed",
        "How much trust the legal system?",
        "Income quartiles",
        "Rural/urban living",
        "Age"
    ],

    "The value of what I do is not recognised by others?": [
        "Employment - 7 groups",
        "How frequently participate in social activities?",
        "A person to get support from when feeling depressed",
        "Education completed",
        "Marital status"
    ],

    "Marital status": [
        "Age",
        "Household structure",
        "How frequently participate in social activities?",
        "No. of children",
        "Education completed"
    ],

    "Can most people be trusted?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the press?",
        "How much trust the government?",
        "Education completed",
        "Rural/urban living"
    ]
}

### Load data, recode variables, and define helper function

In [ ]:
# Data
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

# Reversing the scale on some features (higher values = more positive)
data["Health condition"] = 6 - data["Health condition"]
data["Can't find the way because life has become so complicated?"] = 6 - data["Can't find the way because life has become so complicated?"]
data["Household able to make ends meet?"] = 7 - data["Household able to make ends meet?"]
data["A person to get support from when feeling depressed"] = 3 - data["A person to get support from when feeling depressed"]
data["I feel I am free to decide how to live my life"] = 6 - data["I feel I am free to decide how to live my life"]
data["I generally feel that what I do in life is worthwhile"] = 6 - data["I generally feel that what I do in life is worthwhile"]
data["I am optimistic about the future"] = 6 - data["I am optimistic about the future"]

# Binary marital status: 1 = married/living with partner, 0 = all other categories
data["Marital status"] = (data["Marital status"] == 1).astype(int)

# Age subgroup variable
age_column = "Age"

# Important: Age is the subgroup variable here.
# Therefore, Age is removed from the confounder sets within the age-specific models,
# because it is constant inside each subgroup.
def clean_confounders_for_age_subgroups(treatment, confounders):
    return [c for c in confounders if c not in [age_column, treatment]]


def calculate_causal_values(data_source, confounder_map):
    # Save results
    results = []

    # Iterate through all features
    for treatment, confounders in confounder_map.items():
        adjusted_confounders = clean_confounders_for_age_subgroups(treatment, confounders)

        try:
            print(f"Treatment: {treatment}")
            model = CausalModel(
                data=data_source,
                treatment=treatment,
                outcome=outcome,
                common_causes=adjusted_confounders
            )

            identified_estimand = model.identify_effect()

            estimate = model.estimate_effect(
                identified_estimand,
                method_name="backdoor.linear_regression"
            )

            results.append({
                "Treatment": treatment,
                "ATE (DML)": estimate.value,
                "Confounders": ", ".join(adjusted_confounders)
            })

        except Exception as e:
            results.append({
                "Treatment": treatment,
                "ATE (DML)": None,
                "Confounders": ", ".join(adjusted_confounders),
                "Error": str(e)
            })

    return results

### Create age subgroups

In [ ]:
# Age subgroups
df_age1 = data[data[age_column] == 1].copy()
df_age2 = data[data[age_column] == 2].copy()
df_age3 = data[data[age_column] == 3].copy()
df_age4 = data[data[age_column] == 4].copy()
df_age5 = data[data[age_column] == 5].copy()

age_subgroups = {
    "age1": df_age1,
    "age2": df_age2,
    "age3": df_age3,
    "age4": df_age4,
    "age5": df_age5,
}

for group_name, group_df in age_subgroups.items():
    print(group_name, group_df.shape)

### Calculate ATEs for each age subgroup

In [ ]:
age_results = {}

for group_name, group_df in age_subgroups.items():
    print(f"\n===== {group_name} =====")
    df_results = pd.DataFrame(calculate_causal_values(group_df, feature_confounder_map))
    df_results.sort_values(by="ATE (DML)", ascending=False, inplace=True, key=abs)
    age_results[group_name] = df_results
    display(df_results)

### Combined Results

In [ ]:
# Create one comparison table with one ATE column per age subgroup
comparison_tables = []

for group_name, df_results in age_results.items():
    temp = df_results[["Treatment", "ATE (DML)"]].rename(
        columns={"ATE (DML)": f"ATE_{group_name}"}
    )
    comparison_tables.append(temp)

# Merge all age-specific result tables
from functools import reduce

df_compare_age = reduce(
    lambda left, right: pd.merge(left, right, on="Treatment", how="outer"),
    comparison_tables
)

# Sort by absolute ATE in age1 as a simple default
# You can change this to another subgroup or to the mean absolute ATE.
ate_columns = [col for col in df_compare_age.columns if col.startswith("ATE_")]
df_compare_age["Mean_abs_ATE"] = df_compare_age[ate_columns].abs().mean(axis=1)
df_compare_age = df_compare_age.sort_values("Mean_abs_ATE", ascending=False)

display(df_compare_age)

### Export Results

In [ ]:
# Export combined results
df_compare_age.to_csv("Results/results_age_causal_analysis.csv", index=False)

# Optional: export each subgroup table separately
for group_name, df_results in age_results.items():
    df_results.to_csv(f"Results/results_{group_name}_causal_analysis.csv", index=False)